In [1]:
import pandas as pd
from datetime import datetime
import duckdb

Задача: «Лидеры категорий»
Контекст:
Маркетингу нужно выделить лояльных пользователей, которые тратят в конкретной категории значительно больше, чем «средний» покупатель в этой же категории.

Твое задание:
Выведи список пользователей, чьи суммарные траты в конкретной категории выше средних суммарных трат всех пользователей в этой же категории.

Таблицы:

orders (информация о заказах):

order_id (int) — идентификатор заказа;

user_id (int) — идентификатор пользователя;

order_date (date) — дата заказа.

order_items (детализация заказов):

order_id (int) — идентификатор заказа;

product_id (int) — идентификатор товара;

quantity (int) — количество купленного товара.

products (справочник товаров):

product_id (int) — идентификатор товара;

category_name (varchar) — название категории;

price (numeric) — цена за единицу товара.

In [2]:

orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'user_id': [101, 102, 101, 103, 102, 104, 103, 104],
    'order_date': [
        datetime(2023, 1, 15),
        datetime(2023, 1, 16),
        datetime(2023, 1, 17),
        datetime(2023, 1, 18),
        datetime(2023, 1, 19),
        datetime(2023, 1, 20),
        datetime(2023, 1, 21),
        datetime(2023, 1, 22)
    ]
})
order_items = pd.DataFrame({
    'order_id': [1, 1, 2, 3, 3, 4, 5, 6, 7, 8],
    'product_id': [1001, 1002, 1003, 1001, 1004, 1005, 1006, 1007, 1008, 1009],
    'quantity': [2, 1, 3, 1, 2, 4, 2, 1, 3, 2]
})
products = pd.DataFrame({
    'product_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009],
    'category_name': ['Электроника', 'Электроника', 'Одежда', 'Электроника',
                    'Одежда', 'Продукты', 'Продукты', 'Книги', 'Книги'],
    'price': [15000, 8000, 5000, 12000, 3000, 200, 150, 400, 600]
})

In [9]:
query = """
WITH user_spending AS (
    SELECT
        p.category_name,
        o.user_id,
        SUM(oi.quantity * p.price) AS total_user_category_spent
    FROM orders o
    JOIN order_items oi USING(order_id)
    JOIN products p USING(product_id)
    GROUP BY 1, 2
)
SELECT 
    user_id,
    category_name,
    total_user_category_spent
FROM (
    SELECT *, 
           AVG(total_user_category_spent) OVER(PARTITION BY category_name) as avg_spent_cat
    FROM user_spending
) t
WHERE total_user_category_spent > avg_spent_cat;
"""
result = duckdb.query(query).to_df()
result

,user_id,category_name,total_user_category_spent
0,102,Одежда,15000.0
1,102,Продукты,400.0


In [22]:
df = orders.merge(order_items).merge(products)
df['summ'] = df['quantity'] * df['price']
df = df.groupby(['category_name', 'user_id'], as_index=False).agg(total_user_category_spent=('summ', 'sum'))
df['avg_spent_cat'] = df.groupby('category_name').total_user_category_spent.transform('mean')
df[df['total_user_category_spent'] > df['avg_spent_cat']][['user_id', 'category_name', 'total_user_category_spent']]

,user_id,category_name,total_user_category_spent
2,102,Одежда,15000
4,102,Продукты,400


In [ ]:
# 1. Заранее оставляем только нужные колонки
items = order_items[['order_id', 'product_id', 'quantity', 'price']]
prods = products[['product_id', 'category_name']]
ords = orders[['order_id', 'user_id']]

# 2. Считаем стоимость сразу в order_items, чтобы не тащить лишнее через мерджи
items['summ'] = items['quantity'] * items['price']

# 3. Последовательно объединяем и группируем
# Здесь мы избавляемся от таблицы orders как можно раньше
res = items.merge(prods, on='product_id').merge(ords, on='order_id')

# 4. Группировка
df = res.groupby(['category_name', 'user_id'], as_index=False)['summ'].sum()
df.rename(columns={'summ': 'total_user_category_spent'}, inplace=True)

# 5. Оконная функция и фильтр
df['avg_spent_cat'] = df.groupby('category_name')['total_user_category_spent'].transform('mean')
result = df[df['total_user_category_spent'] > df['avg_spent_cat']]

Задача: «Retention первого месяца»
Контекст:
Нам нужно понять, насколько качественный трафик пришел к нам в январе 2024 года. Для этого необходимо рассчитать Retention 1-го месяца (какой процент пользователей, зарегистрировавшихся в январе, совершил хотя бы одно действие в феврале).

Твое задание:
Напиши запрос, который выведет одну строку с тремя колонками:

Количество пользователей, зарегистрировавшихся в январе 2024 года.

Количество этих же пользователей, которые совершили хотя бы одно действие в феврале 2024 года.

Процентное соотношение (Retention Rate), округленное до двух знаков после запятой.

Таблицы:

users (регистрации):

user_id (int) — идентификатор пользователя;

signup_date (date) — дата регистрации.

user_actions (активность):

action_id (int) — идентификатор действия;

user_id (int) — идентификатор пользователя;

action_date (date) — дата совершения действия.

Ожидаемый результат:
Колонки: jan_cohort_size, feb_returned_users, retention_rate.

In [14]:
users_data = {
    'user_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'signup_date': [
        '2024-01-05', '2024-01-12', '2024-01-25', '2024-01-30', '2024-01-31', # Январские
        '2023-12-25', '2023-12-30',                                           # Декабрьские
        '2024-02-01', '2024-02-10', '2024-02-15'                              # Февральские
    ]
}
users = pd.DataFrame(users_data)
users['signup_date'] = pd.to_datetime(users['signup_date'])
actions_data = {
    'action_id': range(1, 16),
    'user_id': [
        1, 1, 2, 4, 4,  # Активность январских в январе
        1, 2, 2,        # Активность январских в феврале (наш таргет)
        6, 7,           # Активность декабрьских в январе
        8, 9, 10,       # Активность февральских в феврале
        1, 4            # Активность январских в марте
    ],
    'action_date': [
        '2024-01-06', '2024-01-10', '2024-01-15', '2024-01-30', '2024-01-31',
        '2024-02-05', '2024-02-10', '2024-02-20',
        '2024-01-10', '2024-01-15',
        '2024-02-02', '2024-02-11', '2024-02-16',
        '2024-03-01', '2024-03-05'
    ]
}
user_actions = pd.DataFrame(actions_data)
user_actions['action_date'] = pd.to_datetime(user_actions['action_date'])


In [8]:
query = """
SELECT 
    COUNT(distinct user_id) AS jan_cohort_size,
    COUNT(DISTINCT CASE WHEN action_date >= '2024-02-01' AND action_date < '2024-03-01' THEN user_id END) AS feb_returned_users,
    ROUND(
        COUNT(DISTINCT CASE WHEN action_date >= '2024-02-01' AND action_date < '2024-03-01' THEN user_id END) * 100.0 / 
        NULLIF(COUNT(distinct user_id), 0), 
    2) AS retention_rate
FROM users u
LEFT JOIN user_actions a USING(user_id)
WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-02-01';
"""
result = duckdb.query(query).to_df()
result

,jan_cohort_size,feb_returned_users,retention_rate
0,5,2,40.0


In [29]:
jan_cohort = users[
    (users['signup_date'] >= '2024-01-01') & 
    (users['signup_date'] < '2024-02-01')
]
jan_cohort_size = jan_cohort['user_id'].nunique()
merged_data = jan_cohort.merge(user_actions, how='left')
feb_returned_users = merged_data[
    (merged_data['action_date'] >= '2024-02-01') & 
    (merged_data['action_date'] < '2024-03-01')
]['user_id'].nunique()
retention_rate = round((feb_returned_users * 100.0 / jan_cohort_size), 2) if jan_cohort_size > 0 else 0
result = pd.DataFrame({
    'jan_cohort_size': [jan_cohort_size],
    'feb_returned_users': [feb_returned_users],
    'retention_rate': [retention_rate]
})
result

,jan_cohort_size,feb_returned_users,retention_rate
0,5,2,40.0
